In [ ]:
# stream video from default video (sender)

alireza@leris-lab:~$ gst-launch-1.0 -v videotestsrc !     
video/x-raw,width=1280,height=720,framerate=30/1 !     
x264enc tune=zerolatency bitrate=2000 speed-preset=ultrafast !     
rtph264pay config-interval=1 pt=96 !     udpsink host=127.0.0.1 port=5000


In [ ]:
# stream video from Webcam (sender)


alireza@leris-lab:~$ gst-launch-1.0 -v v4l2src device=/dev/video0 !     videoconvert !     
video/x-raw,width=640,height=480,framerate=30/1 !     
x264enc tune=zerolatency bitrate=1500 speed-preset=ultrafast !     
rtph264pay config-interval=1 pt=96 !     udpsink host=127.0.0.1 port=5000


# Note: Customization: Change bitrate=2000 to 500 to see how the quality degrades, 
or change tune=zerolatency to see how it affects the "live" feel.

In [ ]:
# Receiver & Display (Receiver)

alireza@leris-lab:~$ gst-launch-1.0 -v udpsrc port=5000 caps="application/x-rtp, media=(string)video, 
clock-rate=(int)90000, encoding-name=(string)H264, payload=(int)96" !     
rtpjitterbuffer latency=100 !     rtph264depay !     avdec_h264 !     videoconvert !     autovideosink sync=false


## Note: Customization: Adjust latency=100 in the rtpjitterbuffer. 
If you set it to 0, the video might stutter if your CPU spikes. 
If you set it to 2000, the video will be smooth but "behind" the sender by 2 seconds.

Element,Function,Key Options & Settings
v4l2src,Source: Grabs frames from Linux camera.,"device: Path to camera (e.g., /dev/video0).num-buffers: Stop after X frames."
videoconvert,Converter: Changes pixel formats.,Usually automatic; ensures camera output fits encoder input.
x264enc,Codec: Compresses the video (Spatial/Temporal).,"bitrate: Target speed in kbps (e.g., 2000).tune=zerolatency: Removes delay for real-time.speed-preset: ultrafast (low CPU) to placebo (high CPU).key-int-max: Distance between I-frames (lower = faster recovery from errors)."
rtph264pay,Packetizer: Chops video into RTP packets.,config-interval: Sends H.264 headers frequently (so viewers can join mid-stream).pt: Payload type (ID for the stream).
udpsink,Protocol: The transport truck.,"host: Destination IP.port: Destination port.sync: If true, waits for the clock (standard); if false, sends as fast as possible."